# Random Forest Model



In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [3]:
X = train_df.drop(['id', 'Heart Disease'], axis=1)
y = train_df['Heart Disease']

X_test_final = test_df.drop(['id'], axis=1)

# Feature Engineering
X['Age_MaxHR'] = X['Age'] * X['Max HR']
X['BP_Cholesterol_Ratio'] = X['BP'] / (X['Cholesterol'] + 1)
X['Risk_Score'] = X['Age'] + X['BP'] + (X['Cholesterol'] / 2) - X['Max HR']
X['ST_interaction'] = X['ST depression'] * X['Slope of ST']

X_test_final['Age_MaxHR'] = X_test_final['Age'] * X_test_final['Max HR']
X_test_final['BP_Cholesterol_Ratio'] = X_test_final['BP'] / (X_test_final['Cholesterol'] + 1)
X_test_final['Risk_Score'] = X_test_final['Age'] + X_test_final['BP'] + (X_test_final['Cholesterol'] / 2) - X_test_final['Max HR']
X_test_final['ST_interaction'] = X_test_final['ST depression'] * X_test_final['Slope of ST']

categorical_cols = [col for col in X.columns if X[col].dtype == 'object']
numerical_cols = [col for col in X.columns if X[col].dtype in ['int64', 'float64']]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [4]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [5]:
X_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)

pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('model', rf_model)])

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 15],
    'model__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=StratifiedKFold(n_splits=3), scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
best_model_rf = grid_search.best_estimator_

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best parameters: {'model__max_depth': 15, 'model__min_samples_split': 2, 'model__n_estimators': 200}


In [6]:
y_pred = best_model_rf.predict(X_val)
print(f"Accuracy: {accuracy_score(y_val, y_pred)}")
print(classification_report(y_val, y_pred))

Accuracy: 0.8864920634920634
              precision    recall  f1-score   support

           0       0.89      0.90      0.90     69509
           1       0.88      0.86      0.87     56491

    accuracy                           0.89    126000
   macro avg       0.89      0.88      0.89    126000
weighted avg       0.89      0.89      0.89    126000



In [ ]:
test_preds_rf = best_model_rf.predict(X_test_final)

submission_rf = pd.DataFrame({'id': test_df['id'], 'Heart Disease': test_preds_rf})
submission_rf.to_csv('submission_rf.csv', index=False)
print("Saved submission_rf.csv")

Saved submission_rf.csv
